# Tomato Leaf Disease Classification - Local CPU Version

Features: HOG + ORB (Bag of Visual Words: raw and TF-IDF).
Models: SVM and KNN. Runs on CPU, no GPU required.

Pipeline:
1. Train 8 base models (SVM/KNN x 4 feature sets) and compare.
2. Build ensembles (Voting, Stacking) and compare against the best base model.

In [17]:
import os, cv2, json, zipfile, shutil, warnings, time, copy, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
from collections import Counter
from tqdm import tqdm
warnings.filterwarnings('ignore')

from skimage.feature import hog
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import MiniBatchKMeans
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

print('All libraries imported.')

All libraries imported.


In [18]:
# ==============================================================================
#  CONFIGURATION - EDIT THESE PATHS
# ==============================================================================

DATASET_DIR = Path(r'c:\BINUS SOCS\Semester 4\comvis\aol\PlantVillage-Dataset-master')
OUTPUT_DIR = Path('./tomato_output')
MODELS_DIR = OUTPUT_DIR / 'models'
FIGURES_DIR = OUTPUT_DIR / 'reports' / 'figures'
RESULTS_DIR = OUTPUT_DIR / 'reports' / 'results'
PROCESSED_DIR = OUTPUT_DIR / 'data' / 'processed'

for d in [MODELS_DIR, FIGURES_DIR, RESULTS_DIR, PROCESSED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MAX_PER_CLASS = 1000
RANDOM_STATE = 42
TEST_SIZE = 0.20
N_CV_FOLDS = 5

CLASS_MAP = {
    'Healthy':      ['healthy'],
    'Early_Blight': ['Early_blight', 'Early_Blight'],
    'Late_Blight':  ['Late_blight', 'Late_Blight'],
}
CLASS_INFO = {
    'Healthy':      {'color': '#4CAF50', 'desc': 'No visible disease.'},
    'Early_Blight': {'color': '#FF7043', 'desc': 'Alternaria solani. Dark concentric ring spots.'},
    'Late_Blight':  {'color': '#FFA726', 'desc': 'Phytophthora infestans. Dark water-soaked lesions.'},
}

IMG_SIZE = (128, 128)
HOG_ORIENTATIONS = 9
HOG_PPC = (8, 8)
HOG_CPB = (2, 2)
ORB_N_FEATURES = 300
BOVW_VOCAB_SIZE = 500

# Base model hyperparameters
SVM_C = 10
SVM_GAMMA = 'scale'
KNN_K = 5
KNN_METRIC = 'cosine'
KNN_WEIGHTS = 'distance'

print('Config ready.')
print(f'   Dataset dir: {DATASET_DIR}')
print(f'   Output dir : {OUTPUT_DIR.absolute()}')

Config ready.
   Dataset dir: c:\BINUS SOCS\Semester 4\comvis\aol\PlantVillage-Dataset-master
   Output dir : c:\BINUS SOCS\Semester 4\comvis\aol\tomato_output


In [19]:
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

if not DATASET_DIR.exists():
    raise FileNotFoundError(f'DATASET_DIR not found: {DATASET_DIR}')

all_dirs = [d for d in DATASET_DIR.rglob('*') if d.is_dir()]
TARGET_FOLDERS = {cls: None for cls in CLASS_MAP}
for d in all_dirs:
    for cls, keywords in CLASS_MAP.items():
        if TARGET_FOLDERS[cls] is None and any(kw in d.name for kw in keywords) and 'Tomato' in d.name:
            TARGET_FOLDERS[cls] = d

for cls, path in TARGET_FOLDERS.items():
    if path:
        imgs = list(path.glob('*.jpg')) + list(path.glob('*.JPG')) + list(path.glob('*.jpeg')) + list(path.glob('*.png'))
        print(f'  {cls}: {len(imgs)} images -> {path.name}')
    else:
        raise RuntimeError(f'Missing folder for: {cls}')

records = []
for cls, folder in TARGET_FOLDERS.items():
    paths = []
    for ext in ('*.jpg', '*.JPG', '*.jpeg', '*.png', '*.PNG'):
        paths.extend(folder.glob(ext))
    if len(paths) > MAX_PER_CLASS:
        paths = random.sample(paths, MAX_PER_CLASS)
    for p in paths:
        records.append({'path': str(p), 'label': cls})

df = pd.DataFrame(records)
print(f'\nTotal: {len(df)}')
print(df['label'].value_counts().to_string())

train_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df['label'])
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
train_df.to_csv(PROCESSED_DIR / 'train.csv', index=False)
test_df.to_csv(PROCESSED_DIR / 'test.csv', index=False)
print(f'Train: {len(train_df)} | Test: {len(test_df)}')

  Healthy: 3180 images -> Tomato___healthy
  Early_Blight: 2000 images -> Tomato___Early_blight
  Late_Blight: 3817 images -> Tomato___Late_blight

Total: 3000
label
Healthy         1000
Early_Blight    1000
Late_Blight     1000
Train: 2400 | Test: 600


In [20]:
counts = df['label'].value_counts()
colors = [CLASS_INFO[k]['color'] for k in counts.index]
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(counts.index, counts.values, color=colors, edgecolor='white')
ax.bar_label(bars, padding=4, fontsize=11, fontweight='bold')
ax.set_title('Class Distribution', fontsize=14, fontweight='bold')
ax.set_ylabel('Images')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'class_distribution.png', dpi=150)
plt.close()

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for row, cls in enumerate(CLASS_MAP.keys()):
    sp = df[df['label'] == cls]['path'].sample(4, random_state=7).tolist()
    for col, p in enumerate(sp):
        img = cv2.imread(p)
        if img is not None:
            axes[row, col].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(cls.replace('_', ' '), fontsize=11,
                                     fontweight='bold', color=CLASS_INFO[cls]['color'])
plt.suptitle('Sample Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'sample_images.png', dpi=150)
plt.close()
print('EDA saved.')

EDA saved.


In [5]:
def preprocess_image(path, img_size=(128, 128), use_clahe=True):
    bgr = cv2.imread(str(path))
    if bgr is None:
        return None
    resized = cv2.resize(bgr, img_size)
    gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
    gray_norm = gray.astype(np.float32) / 255.0
    out = {'bgr': resized, 'gray': gray_norm}
    if use_clahe:
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        out['gray_clahe'] = clahe.apply(gray).astype(np.float32) / 255.0
    else:
        out['gray_clahe'] = gray_norm
    return out

print('Preprocessing ready.')

Preprocessing ready.


In [6]:
def extract_hog(gray_norm, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2)):
    return hog(gray_norm, orientations=orientations,
               pixels_per_cell=pixels_per_cell, cells_per_block=cells_per_block,
               visualize=False, feature_vector=True)

def extract_orb_desc(gray_norm, n_features=300):
    gray_u8 = (gray_norm * 255).astype(np.uint8)
    orb = cv2.ORB_create(nfeatures=n_features)
    _, desc = orb.detectAndCompute(gray_u8, None)
    return desc

def build_bovw_codebook(train_paths, img_size, orb_n, vocab_size, random_state):
    all_desc = []
    for p in tqdm(train_paths, desc='ORB descriptors'):
        proc = preprocess_image(p, img_size)
        if proc is None:
            continue
        desc = extract_orb_desc(proc['gray'], orb_n)
        if desc is not None:
            all_desc.append(desc.astype(np.float32))
    if not all_desc:
        return None
    all_desc = np.vstack(all_desc)
    print(f'  Total descriptors: {all_desc.shape[0]:,}')
    kmeans = MiniBatchKMeans(n_clusters=vocab_size, random_state=random_state,
                              n_init=10, batch_size=10000)
    kmeans.fit(all_desc)
    print(f'  Codebook fitted ({vocab_size} visual words)')
    return kmeans

def extract_bovw(gray_norm, codebook, orb_n):
    desc = extract_orb_desc(gray_norm, orb_n)
    if desc is None or len(desc) == 0:
        return np.zeros(codebook.n_clusters, dtype=np.float32)
    words = codebook.predict(desc.astype(np.float32))
    hist, _ = np.histogram(words, bins=np.arange(codebook.n_clusters + 1))
    hist = hist.astype(np.float32)
    if hist.sum() > 0:
        hist /= hist.sum()
    return hist

def extract_bovw_tfidf(gray_norm, codebook, orb_n, idf_weights):
    hist = extract_bovw(gray_norm, codebook, orb_n)
    tfidf = hist * idf_weights
    norm = np.linalg.norm(tfidf)
    return tfidf / norm if norm > 0 else tfidf

def compute_idf(train_paths, img_size, codebook, orb_n):
    doc_count = 0
    df_count = np.zeros(codebook.n_clusters)
    for p in tqdm(train_paths, desc='Computing IDF'):
        proc = preprocess_image(p, img_size)
        if proc is None:
            continue
        h = extract_bovw(proc['gray'], codebook, orb_n)
        df_count += (h > 0).astype(np.float32)
        doc_count += 1
    return (np.log((doc_count + 1) / (df_count + 1)) + 1).astype(np.float32)

print('Feature extractors ready.')

Feature extractors ready.


In [7]:
train_paths = train_df['path'].tolist()
codebook = build_bovw_codebook(train_paths, IMG_SIZE, ORB_N_FEATURES, BOVW_VOCAB_SIZE, RANDOM_STATE)
joblib.dump(codebook, MODELS_DIR / 'orb_codebook.joblib')
idf_weights = compute_idf(train_paths, IMG_SIZE, codebook, ORB_N_FEATURES)
np.save(MODELS_DIR / 'idf_weights.npy', idf_weights)
print(f'IDF shape: {idf_weights.shape}')

ORB descriptors: 100%|██████████| 2400/2400 [00:17<00:00, 136.28it/s]


  Total descriptors: 277,213
  Codebook fitted (500 visual words)


Computing IDF: 100%|██████████| 2400/2400 [00:16<00:00, 143.73it/s]

IDF shape: (500,)


In [8]:
def build_features(df_split, img_size, hog_orient, hog_ppc, hog_cpb, codebook, orb_n, idf_weights):
    X_hog, X_orb_raw, X_orb_tfidf, X_comb, y_list = [], [], [], [], []
    for _, row in tqdm(df_split.iterrows(), total=len(df_split), desc='Extracting'):
        proc = preprocess_image(row['path'], img_size)
        if proc is None:
            continue
        h = extract_hog(proc['gray_clahe'], orientations=hog_orient,
                        pixels_per_cell=hog_ppc, cells_per_block=hog_cpb)
        o_raw = extract_bovw(proc['gray'], codebook, orb_n)
        o_tfidf = extract_bovw_tfidf(proc['gray'], codebook, orb_n, idf_weights)
        X_hog.append(h)
        X_orb_raw.append(o_raw)
        X_orb_tfidf.append(o_tfidf)
        X_comb.append(np.concatenate([h, o_tfidf]))
        y_list.append(row['label'])
    return (
        np.array(X_hog, dtype=np.float32),
        np.array(X_orb_raw, dtype=np.float32),
        np.array(X_orb_tfidf, dtype=np.float32),
        np.array(X_comb, dtype=np.float32),
        np.array(y_list)
    )

X_hog_tr, X_orb_tr, X_orb_tfidf_tr, X_comb_tr, y_tr_raw = build_features(
    train_df, IMG_SIZE, HOG_ORIENTATIONS, HOG_PPC, HOG_CPB,
    codebook, ORB_N_FEATURES, idf_weights)
X_hog_te, X_orb_te, X_orb_tfidf_te, X_comb_te, y_te_raw = build_features(
    test_df, IMG_SIZE, HOG_ORIENTATIONS, HOG_PPC, HOG_CPB,
    codebook, ORB_N_FEATURES, idf_weights)

le = LabelEncoder()
le.fit(y_tr_raw)
y_tr = le.transform(y_tr_raw)
y_te = le.transform(y_te_raw)
joblib.dump(le, MODELS_DIR / 'label_encoder.joblib')
label_names = list(le.classes_)

print(f'Classes: {le.classes_}')
print(f'HOG: {X_hog_tr.shape[1]} | ORB raw: {X_orb_tr.shape[1]} | ORB TF-IDF: {X_orb_tfidf_tr.shape[1]} | Combined: {X_comb_tr.shape[1]}')
print(f'Train: {len(y_tr)} | Test: {len(y_te)}')

Extracting: 100%|██████████| 600/600 [00:03<00:00, 152.95it/s]

Classes: ['Early_Blight' 'Healthy' 'Late_Blight']
HOG: 8100 | ORB raw: 500 | ORB TF-IDF: 500 | Combined: 8600
Train: 2400 | Test: 600


## Part 1: Train and Compare 8 Base Models

We train two classifiers (SVM and KNN) on four feature sets:

- HOG
- ORB raw (Bag of Visual Words histogram)
- ORB TF-IDF (weighted Bag of Visual Words)
- Combined (HOG + ORB TF-IDF)

This gives 8 base models. Each pipeline is `StandardScaler -> classifier`.

In [9]:
def evaluate_model(model, X_te, y_te, label_names, title=''):
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_te, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_te, y_pred, average='macro', zero_division=0)
    print(f"\n{'='*58}\n {title}\n{'='*58}")
    print(f'  Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f}')
    print(classification_report(y_te, y_pred, target_names=label_names))
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1,
            'y_pred': y_pred, 'title': title}

feature_splits = {
    'HOG': (X_hog_tr, X_hog_te),
    'ORB_raw': (X_orb_tr, X_orb_te),
    'ORB_TFIDF': (X_orb_tfidf_tr, X_orb_tfidf_te),
    'Combined': (X_comb_tr, X_comb_te)
}

results = {}
trained_models = {}

for feat_name, (X_tr_f, X_te_f) in feature_splits.items():
    for model_name, factory in [
        ('SVM', lambda: Pipeline([
            ('scaler', StandardScaler()),
            ('clf', SVC(kernel='rbf', C=SVM_C, gamma=SVM_GAMMA,
                        probability=True, random_state=RANDOM_STATE))
        ])),
        ('KNN', lambda: Pipeline([
            ('scaler', StandardScaler()),
            ('clf', KNeighborsClassifier(n_neighbors=KNN_K, metric=KNN_METRIC,
                                         weights=KNN_WEIGHTS, n_jobs=-1))
        ]))]:
        key = f'{model_name} + {feat_name}'
        print(f'\nTraining {key} ...')
        pipe = factory()
        pipe.fit(X_tr_f, y_tr)
        results[key] = evaluate_model(pipe, X_te_f, y_te, label_names, key)
        results[key]['X_te'] = X_te_f
        trained_models[key] = pipe

print(f'\n{len(results)} base models trained.')


Training SVM + HOG ...

 SVM + HOG
  Acc: 0.8950 | Prec: 0.8942 | Rec: 0.8950 | F1: 0.8943
              precision    recall  f1-score   support

Early_Blight       0.87      0.82      0.84       200
     Healthy       0.94      0.97      0.96       200
 Late_Blight       0.87      0.90      0.88       200

    accuracy                           0.90       600
   macro avg       0.89      0.90      0.89       600
weighted avg       0.89      0.90      0.89       600


Training KNN + HOG ...

 KNN + HOG
  Acc: 0.8867 | Prec: 0.8858 | Rec: 0.8867 | F1: 0.8851
              precision    recall  f1-score   support

Early_Blight       0.87      0.79      0.83       200
     Healthy       0.90      0.97      0.94       200
 Late_Blight       0.88      0.90      0.89       200

    accuracy                           0.89       600
   macro avg       0.89      0.89      0.89       600
weighted avg       0.89      0.89      0.89       600


Training SVM + ORB_raw ...

 SVM + ORB_raw
  Acc: 0.6

In [10]:
base_summary = pd.DataFrame([
    {
        'Model + Features': k,
        'Accuracy': round(v['accuracy'], 4),
        'Precision': round(v['precision'], 4),
        'Recall': round(v['recall'], 4),
        'F1-Score': round(v['f1'], 4),
    }
    for k, v in results.items()
]).sort_values('F1-Score', ascending=False).reset_index(drop=True)

print('=' * 70)
print('  BASE MODEL COMPARISON (8 models, sorted by macro F1)')
print('=' * 70)
print(base_summary.to_string(index=False))
base_summary.to_csv(RESULTS_DIR / 'base_model_comparison.csv', index=False)

best_base_key = base_summary.iloc[0]['Model + Features']
best_base_f1 = base_summary.iloc[0]['F1-Score']
print(f'\nBest base model: {best_base_key} (F1 = {best_base_f1:.4f})')

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(base_summary))
w = 0.2
for i, (m, c) in enumerate(zip(['Accuracy', 'Precision', 'Recall', 'F1-Score'],
                               ['#1565C0', '#2E7D32', '#FF6F00', '#C62828'])):
    ax.bar(x + i*w, base_summary[m], w, label=m, color=c, alpha=0.85)
ax.set_xticks(x + 1.5*w)
ax.set_xticklabels(base_summary['Model + Features'], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Score')
ax.set_title('8 Base Models Performance', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
ax.set_ylim(0.4, 1.0)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'base_model_comparison.png', dpi=150)
plt.close()
print('Base comparison chart saved.')

  BASE MODEL COMPARISON (8 models, sorted by macro F1)
Model + Features  Accuracy  Precision  Recall  F1-Score
  SVM + Combined    0.8983     0.8976  0.8983    0.8975
       SVM + HOG    0.8950     0.8942  0.8950    0.8943
  KNN + Combined    0.8883     0.8872  0.8883    0.8871
       KNN + HOG    0.8867     0.8858  0.8867    0.8851
 SVM + ORB_TFIDF    0.6950     0.6961  0.6950    0.6951
   SVM + ORB_raw    0.6900     0.6884  0.6900    0.6883
 KNN + ORB_TFIDF    0.6367     0.6389  0.6367    0.6372
   KNN + ORB_raw    0.6300     0.6349  0.6300    0.6305

Best base model: SVM + Combined (F1 = 0.8975)
Base comparison chart saved.


## Part 2: Ensemble Methods

Now we build ensembles on the Combined feature set and compare them against the
best base model found in Part 1.

- **Voting (soft)**: averages the predicted probabilities of SVM and KNN.
- **Stacking**: trains a meta-learner (Logistic Regression) on top of the SVM and
  KNN predictions, using internal cross-validation to avoid leakage.

Ensembles can improve accuracy but are heavier to train and deploy. The final
comparison shows whether the extra complexity is worth it.

In [11]:
scaler_ens = StandardScaler()
X_tr_s = scaler_ens.fit_transform(X_comb_tr)
X_te_s = scaler_ens.transform(X_comb_te)

base_svm = SVC(kernel='rbf', C=SVM_C, gamma=SVM_GAMMA, probability=True, random_state=RANDOM_STATE)
base_knn = KNeighborsClassifier(n_neighbors=KNN_K, metric=KNN_METRIC, weights=KNN_WEIGHTS, n_jobs=-1)

ensemble_results = {}
ensemble_models = {}

# Voting (soft)
print('Training Voting (soft) ...')
voting = VotingClassifier(
    estimators=[('svm', base_svm), ('knn', base_knn)],
    voting='soft', n_jobs=-1
)
voting.fit(X_tr_s, y_tr)
yv = voting.predict(X_te_s)
ensemble_results['Voting (soft)'] = {
    'accuracy': accuracy_score(y_te, yv),
    'precision': precision_score(y_te, yv, average='macro', zero_division=0),
    'recall': recall_score(y_te, yv, average='macro', zero_division=0),
    'f1': f1_score(y_te, yv, average='macro', zero_division=0),
    'y_pred': yv
}
ensemble_models['Voting (soft)'] = voting
print(f"  Voting -> Acc: {ensemble_results['Voting (soft)']['accuracy']:.4f}, F1: {ensemble_results['Voting (soft)']['f1']:.4f}")

# Stacking
print('\nTraining Stacking ...')
stacking = StackingClassifier(
    estimators=[
        ('svm', SVC(kernel='rbf', C=SVM_C, gamma=SVM_GAMMA, probability=True, random_state=RANDOM_STATE)),
        ('knn', KNeighborsClassifier(n_neighbors=KNN_K, metric=KNN_METRIC, weights=KNN_WEIGHTS, n_jobs=-1))
    ],
    final_estimator=LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_STATE),
    cv=N_CV_FOLDS, n_jobs=-1
)
stacking.fit(X_tr_s, y_tr)
ys = stacking.predict(X_te_s)
ensemble_results['Stacking'] = {
    'accuracy': accuracy_score(y_te, ys),
    'precision': precision_score(y_te, ys, average='macro', zero_division=0),
    'recall': recall_score(y_te, ys, average='macro', zero_division=0),
    'f1': f1_score(y_te, ys, average='macro', zero_division=0),
    'y_pred': ys
}
ensemble_models['Stacking'] = stacking
print(f"  Stacking -> Acc: {ensemble_results['Stacking']['accuracy']:.4f}, F1: {ensemble_results['Stacking']['f1']:.4f}")
print('\nEnsembles trained.')

Training Voting (soft) ...
  Voting -> Acc: 0.9067, F1: 0.9058

Training Stacking ...
  Stacking -> Acc: 0.8983, F1: 0.8979

Ensembles trained.


In [12]:
# Combine best base model with the two ensembles for the final comparison
final_entries = {}
final_entries[best_base_key] = {
    'accuracy': results[best_base_key]['accuracy'],
    'precision': results[best_base_key]['precision'],
    'recall': results[best_base_key]['recall'],
    'f1': results[best_base_key]['f1'],
    'y_pred': results[best_base_key]['y_pred']
}
final_entries['Voting (soft)'] = ensemble_results['Voting (soft)']
final_entries['Stacking'] = ensemble_results['Stacking']

final_summary = pd.DataFrame([
    {
        'Model': k,
        'Accuracy': round(v['accuracy'], 4),
        'Precision': round(v['precision'], 4),
        'Recall': round(v['recall'], 4),
        'F1-Score': round(v['f1'], 4),
    }
    for k, v in final_entries.items()
]).sort_values('F1-Score', ascending=False).reset_index(drop=True)

print('=' * 70)
print('  FINAL COMPARISON: Best Base Model vs Voting vs Stacking')
print('=' * 70)
print(final_summary.to_string(index=False))
final_summary.to_csv(RESULTS_DIR / 'final_comparison.csv', index=False)

overall_best_key = final_summary.iloc[0]['Model']
overall_best_f1 = final_summary.iloc[0]['F1-Score']
print(f'\nOverall best: {overall_best_key} (F1 = {overall_best_f1:.4f})')

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(final_summary))
w = 0.2
for i, (m, c) in enumerate(zip(['Accuracy', 'Precision', 'Recall', 'F1-Score'],
                               ['#1565C0', '#2E7D32', '#FF6F00', '#C62828'])):
    ax.bar(x + i*w, final_summary[m], w, label=m, color=c, alpha=0.85)
ax.set_xticks(x + 1.5*w)
ax.set_xticklabels(final_summary['Model'], rotation=20, ha='right', fontsize=10)
ax.set_ylabel('Score')
ax.set_title('Best Base Model vs Ensembles', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
ax.set_ylim(0.4, 1.0)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'final_comparison.png', dpi=150)
plt.close()
print('Final comparison chart saved.')

  FINAL COMPARISON: Best Base Model vs Voting vs Stacking
         Model  Accuracy  Precision  Recall  F1-Score
 Voting (soft)    0.9067     0.9066  0.9067    0.9058
      Stacking    0.8983     0.8979  0.8983    0.8979
SVM + Combined    0.8983     0.8976  0.8983    0.8975

Overall best: Voting (soft) (F1 = 0.9058)
Final comparison chart saved.


In [13]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for idx, key in enumerate(final_summary['Model'].tolist()):
    cm = confusion_matrix(y_te, final_entries[key]['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=label_names, yticklabels=label_names)
    axes[idx].set_title(f"{key}\nF1={final_entries[key]['f1']:.3f}", fontsize=10, fontweight='bold')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('True')
plt.suptitle('Confusion Matrices - Finalists', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confusion_matrices_final.png', dpi=150)
plt.close()
print('Confusion matrices saved.')

Confusion matrices saved.


In [14]:
if overall_best_key in trained_models:
    overall_best_model = trained_models[overall_best_key]
elif overall_best_key in ensemble_models:
    overall_best_model = ensemble_models[overall_best_key]
else:
    overall_best_model = None

best_y_pred = final_entries[overall_best_key]['y_pred']
joblib.dump(overall_best_model, MODELS_DIR / 'best_model.joblib')

config = {
    'overall_best': overall_best_key,
    'f1': float(overall_best_f1),
    'best_base_model': best_base_key,
    'img_size': list(IMG_SIZE),
    'hog_orient': HOG_ORIENTATIONS,
    'hog_ppc': list(HOG_PPC),
    'hog_cpb': list(HOG_CPB),
    'orb_n': ORB_N_FEATURES,
    'bovw_vocab': BOVW_VOCAB_SIZE,
    'n_cv_folds': N_CV_FOLDS
}
with open(MODELS_DIR / 'model_config.json', 'w') as f:
    json.dump(config, f, indent=2)

report = classification_report(y_te, best_y_pred, target_names=label_names, digits=4)
with open(RESULTS_DIR / 'classification_report.txt', 'w') as f:
    f.write(f'Overall best: {overall_best_key}\nF1: {overall_best_f1:.4f}\n\n{report}')
print(f'Best model ({overall_best_key}) saved.')
print(report)

Best model (Voting (soft)) saved.
              precision    recall  f1-score   support

Early_Blight     0.8962    0.8200    0.8564       200
     Healthy     0.9515    0.9800    0.9655       200
 Late_Blight     0.8720    0.9200    0.8954       200

    accuracy                         0.9067       600
   macro avg     0.9066    0.9067    0.9058       600
weighted avg     0.9066    0.9067    0.9058       600



In [15]:
print('=' * 70)
print('  FINAL SUMMARY')
print('=' * 70)
print(f'\nDataset: PlantVillage (3 classes, up to {MAX_PER_CLASS}/class)')
print(f'   Train: {len(y_tr)} | Test: {len(y_te)}')
print(f'\nFeatures: HOG, ORB-BoVW (raw + TF-IDF), Combined (HOG + ORB-TFIDF)')
print(f'\nBase models (8): SVM and KNN over 4 feature sets')
print(f'   Best base model: {best_base_key} (F1 = {best_base_f1:.4f})')
print(f'\nEnsembles: Voting (soft), Stacking')
print(f'\nOverall best: {overall_best_key} (F1 = {overall_best_f1:.4f})')
print('\n' + '=' * 70)
print('Final comparison:')
print(final_summary.to_string(index=False))
print(f'\nAll outputs saved to: {OUTPUT_DIR.absolute()}')

  FINAL SUMMARY

Dataset: PlantVillage (3 classes, up to 1000/class)
   Train: 2400 | Test: 600

Features: HOG, ORB-BoVW (raw + TF-IDF), Combined (HOG + ORB-TFIDF)

Base models (8): SVM and KNN over 4 feature sets
   Best base model: SVM + Combined (F1 = 0.8975)

Ensembles: Voting (soft), Stacking

Overall best: Voting (soft) (F1 = 0.9058)

Final comparison:
         Model  Accuracy  Precision  Recall  F1-Score
 Voting (soft)    0.9067     0.9066  0.9067    0.9058
      Stacking    0.8983     0.8979  0.8983    0.8979
SVM + Combined    0.8983     0.8976  0.8983    0.8975

All outputs saved to: c:\BINUS SOCS\Semester 4\comvis\aol\tomato_output


In [16]:
zip_path = OUTPUT_DIR / 'outputs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fp in OUTPUT_DIR.rglob('*'):
        if fp.is_file() and fp.name != 'outputs.zip':
            zf.write(fp, fp.relative_to(OUTPUT_DIR))
print(f'Saved archive: {zip_path}')

Saved archive: tomato_output\outputs.zip
